In [ ]:
# CELL 1
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
CACHE_DIR = "/content/drive/MyDrive/oil_spill_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

if not os.path.exists("train/images"):
    if os.path.exists(f"{CACHE_DIR}/Radar_data.rar"):
        shutil.copy(f"{CACHE_DIR}/Radar_data.rar", "Radar_data.rar")
    else:
        os.system('wget -q "https://zenodo.org/records/4672426/files/Radar_data.rar?download=1" -O Radar_data.rar')
        shutil.copy("Radar_data.rar", f"{CACHE_DIR}/Radar_data.rar")
    os.system('apt-get install -y unrar -qq')
    os.system('unrar x -o+ Radar_data.rar')

if not os.path.exists("oil_spill_pipeline.py"):
    if os.path.exists(f"{CACHE_DIR}/oil_spill_pipeline.py"):
        shutil.copy(f"{CACHE_DIR}/oil_spill_pipeline.py", "oil_spill_pipeline.py")
    else:
        print("Upload oil_spill_pipeline.py now (folder icon, left), then run the line below:")
        print("shutil.copy('oil_spill_pipeline.py', CACHE_DIR + '/oil_spill_pipeline.py')")
print("Setup ready.")

Mounted at /content/drive
Setup ready.


In [ ]:
# CELL 2
from oil_spill_pipeline import *
import joblib

if os.path.exists(f"{CACHE_DIR}/oil_spill_detector.pkl"):
    shutil.copy(f"{CACHE_DIR}/oil_spill_detector.pkl", "oil_spill_detector.pkl")
    detector = joblib.load("oil_spill_detector.pkl")
    print("Detector loaded from cache.")
else:
    print("Training detector (only happens once, takes a minute or two)...")
    df_train, df_val = load_dataset_index(
        "train/dataframe_train_dataset_256_90.csv", "train/dataframe_val_dataset_256_90.csv")
    detector = train_detector(df_train)
    joblib.dump(detector, "oil_spill_detector.pkl")
    shutil.copy("oil_spill_detector.pkl", f"{CACHE_DIR}/oil_spill_detector.pkl")
    print("Trained and cached.")

Detector loaded from cache.


In [ ]:
# CELL 3
!pip install fastapi "uvicorn[standard]" pyngrok nest-asyncio python-multipart -q

In [ ]:
# CELL 4
from oil_spill_pipeline import (
    extract_patch, extract_patch_features, pixel_to_latlon,
    haversine_km, bearing_deg, escalation_tier,
    GLCM_LOW, GLCM_HIGH, _distance_score, _bearing_score, _gap_score,
)
import joblib
detector = joblib.load("oil_spill_detector.pkl")
print("Pipeline and model loaded.")

Pipeline and model loaded.


In [ ]:
'''
shutil.copy('oil_spill_pipeline.py', CACHE_DIR + '/oil_spill_pipeline.py')
print("Saved.")
'''

'\nshutil.copy(\'oil_spill_pipeline.py\', CACHE_DIR + \'/oil_spill_pipeline.py\')\nprint("Saved.")\n'

In [ ]:
# CELL 4b — precompute segmentation heatmaps for all 14 images, ONCE, cached to Drive.
# Run this after Cell 4, before Cell 5. First run will take a while (real computation,
# now with finer stride than before). Every session after, loads instantly from Drive.
import os, shutil

SEGMENT_CACHE_DIR = "segment_cache"
DRIVE_SEGMENT_CACHE = CACHE_DIR + "/segment_cache"
os.makedirs(SEGMENT_CACHE_DIR, exist_ok=True)
os.makedirs(DRIVE_SEGMENT_CACHE, exist_ok=True)

image_files = sorted(f for f in os.listdir("train/images") if f.endswith(".tif"))
print(f"Preparing segmentation heatmaps for {len(image_files)} images...")

for i, name in enumerate(image_files):
    local_path = f"{SEGMENT_CACHE_DIR}/{name}.png"
    drive_path = f"{DRIVE_SEGMENT_CACHE}/{name}.png"

    if os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
        print(f"[{i+1}/{len(image_files)}] {name} — loaded from Drive cache")
        continue

    image = tifffile.imread(f"train/images/{name}")
    # no stride/threshold passed here on purpose — uses the new, fixed defaults
    heatmap = generate_segmentation_heatmap(image, detector)
    save_heatmap_overlay(image, heatmap, local_path)
    shutil.copy(local_path, drive_path)
    print(f"[{i+1}/{len(image_files)}] {name} — computed and cached to Drive")

print("All 14 heatmaps ready.")

Preparing segmentation heatmaps for 14 images...
[1/14] 2018_08_21_.tif — loaded from Drive cache
[2/14] 2018_09_14_.tif — loaded from Drive cache
[3/14] 2018_12_07.tif — loaded from Drive cache
[4/14] 2018_12_07_b.tif — loaded from Drive cache
[5/14] 2018_12_19.tif — loaded from Drive cache
[6/14] 2018_12_19_b.tif — loaded from Drive cache
[7/14] 2018_12_31_b.tif — loaded from Drive cache
[8/14] 20190816.tif — loaded from Drive cache
[9/14] 20190908.tif — loaded from Drive cache
[10/14] 20200224.tif — loaded from Drive cache
[11/14] 20200307.tif — loaded from Drive cache
[12/14] 20200319.tif — loaded from Drive cache
[13/14] 20200331.tif — loaded from Drive cache
[14/14] 20200822.tif — loaded from Drive cache
All 14 heatmaps ready.


In [ ]:
# CELL 5 — backend. Segmentation is now instant: it only ever serves a precomputed file.
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, FileResponse
from pydantic import BaseModel
import numpy as np, io, os, uuid
from PIL import Image

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
IMAGES_DIR = "train/images"
SEGMENT_CACHE_DIR = "segment_cache"   # filled in by Cell 4b, before this cell ever runs
_preview_cache = {}

def to_png_preview(image, max_width=700):
    clipped = np.clip(image, GLCM_LOW, GLCM_HIGH)
    scaled = ((clipped - GLCM_LOW) / (GLCM_HIGH - GLCM_LOW) * 255).astype(np.uint8)
    img = Image.fromarray(scaled)
    ratio = max_width / img.width
    img = img.resize((max_width, int(img.height * ratio)))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    buf.seek(0)
    return buf

def generate_synthetic_fleet(lat, lon, spill_time, seed, n_vessels=5):
    rng = np.random.RandomState(seed)
    fleet = []
    for i in range(n_vessels):
        dist_deg = rng.uniform(0.02, 0.5)
        angle = rng.uniform(0, 360)
        vlat = lat + dist_deg * np.cos(np.radians(angle))
        vlon = lon + dist_deg * np.sin(np.radians(angle))
        has_gap = rng.rand() < 0.35
        fleet.append({
            "vessel_id": f"MV-{rng.randint(1000,9999)}",
            "lat": vlat, "lon": vlon,
            "heading_deg": rng.uniform(0, 360),
            "last_ping_before_gap": spill_time - pd.Timedelta(hours=rng.uniform(1,4)) if has_gap else None,
            "first_ping_after_gap": spill_time + pd.Timedelta(hours=rng.uniform(1,4)) if has_gap else None,
        })
    return fleet

def score_vessel_weighted(vessel, lat, lon, spill_time, weights):
    total = sum(weights.values()) or 1.0
    w = {k: v/total for k, v in weights.items()}
    dist = haversine_km(vessel["lat"], vessel["lon"], lat, lon)
    bearing_to_spill = bearing_deg(vessel["lat"], vessel["lon"], lat, lon)
    d_score = _distance_score(dist)
    b_score = _bearing_score(vessel["heading_deg"], bearing_to_spill)
    g_score = _gap_score(vessel["last_ping_before_gap"], vessel["first_ping_after_gap"], spill_time)
    total_score = w["distance"]*d_score + w["bearing"]*b_score + w["gap"]*g_score
    return {"vessel_id": vessel["vessel_id"], "distance_km": dist, "distance_score": d_score,
            "bearing_score": b_score, "gap_score": g_score, "suspicion_score": total_score}

def run_live_analysis(image, image_path_for_geo, x, y, threshold, weights):
    patch = extract_patch(image, x, y)
    if patch is None:
        return {"status": "ERROR", "reason": "too close to the image edge — click further inside"}
    features = extract_patch_features(patch)
    oil_probability = float(detector.predict_proba(features)[0][1])
    if oil_probability < threshold:
        return {"status": "CLEAN", "oil_probability": oil_probability}

    try:
        lat, lon = pixel_to_latlon(image_path_for_geo, x, y)
        location_is_real = True
    except Exception:
        lat, lon = 28.9, -88.8
        location_is_real = False

    import pandas as pd
    spill_time = pd.Timestamp.utcnow()
    fleet = generate_synthetic_fleet(lat, lon, spill_time, seed=int(x)+int(y))
    scored = sorted((score_vessel_weighted(v, lat, lon, spill_time, weights) for v in fleet),
                     key=lambda s: s["suspicion_score"], reverse=True)
    top = scored[0]["suspicion_score"] if scored else 0.0
    return {"status": "OIL DETECTED", "oil_probability": oil_probability,
            "location": {"lat": lat, "lon": lon, "is_real": location_is_real},
            "escalation_tier": escalation_tier(top), "ranked_suspects": scored}


@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/images")
def list_images():
    out = []
    for name in sorted(os.listdir(IMAGES_DIR)):
        if not name.endswith(".tif"):
            continue
        with rasterio.open(os.path.join(IMAGES_DIR, name)) as src:
            out.append({"name": name, "width": src.width, "height": src.height})
    return out

@app.get("/images/{name}/preview")
def preview(name: str):
    if name not in _preview_cache:
        image = tifffile.imread(os.path.join(IMAGES_DIR, name))
        _preview_cache[name] = to_png_preview(image)
    _preview_cache[name].seek(0)
    return StreamingResponse(_preview_cache[name], media_type="image/png")

# Segmentation — no computation here at all. Cell 4b already made this file; we just hand it back.
@app.get("/images/{name}/segment")
def segment(name: str):
    path = os.path.join(SEGMENT_CACHE_DIR, f"{name}.png")
    if not os.path.exists(path):
        return {"status": "ERROR", "reason": "heatmap not precomputed for this image — did Cell 4b finish running?"}
    return FileResponse(path, media_type="image/png")

class AnalyzeRequest(BaseModel):
    filename: str; x: int; y: int; threshold: float = 0.5
    weight_distance: float = 0.4; weight_bearing: float = 0.3; weight_gap: float = 0.3

@app.post("/analyze")
def analyze(req: AnalyzeRequest):
    path = os.path.join(IMAGES_DIR, req.filename)
    image = tifffile.imread(path)
    weights = {"distance": req.weight_distance, "bearing": req.weight_bearing, "gap": req.weight_gap}
    return run_live_analysis(image, path, req.x, req.y, req.threshold, weights)

@app.post("/analyze_upload")
async def analyze_upload(file: UploadFile = File(...), x: int = Form(...), y: int = Form(...),
                          threshold: float = Form(0.5), weight_distance: float = Form(0.4),
                          weight_bearing: float = Form(0.3), weight_gap: float = Form(0.3)):
    tmp_path = f"/tmp/{uuid.uuid4()}.tif"
    with open(tmp_path, "wb") as f:
        f.write(await file.read())
    image = tifffile.imread(tmp_path)
    weights = {"distance": weight_distance, "bearing": weight_bearing, "gap": weight_gap}
    result = run_live_analysis(image, tmp_path, x, y, threshold, weights)
    os.remove(tmp_path)
    return result

print("API defined. Segmentation now serves precomputed files only.")

API defined. Segmentation now serves precomputed files only.


In [ ]:
# CELL 6 — fixed version, no nest_asyncio needed
from pyngrok import ngrok, conf
import uvicorn

from google.colab import userdata
conf.get_default().auth_token = userdata.get('NGROK_AUTHTOKEN')

public_url = ngrok.connect(8000)
print("PASTE THIS:", public_url.public_url)
print("Backend is live at:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [3012]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


PASTE THIS: https://obscure-frays-sizably.ngrok-free.dev
Backend is live at: NgrokTunnel: "https://obscure-frays-sizably.ngrok-free.dev" -> "http://localhost:8000"
INFO:     2409:40c2:1046:d20c:8000:::0 - "GET / HTTP/1.1" 404 Not Found
INFO:     202.71.156.66:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "GET /health HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "OPTIONS /images HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "GET /health HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "OPTIONS /images HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "GET /images HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "GET /images HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "OPTIONS /images/20200822.tif/segment HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "GET /images/20200822.tif/segment HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "OPTIONS /images/20200331.tif/segment HTTP/1.1" 200 OK
INFO:     202.71.156.66:0 - "GET /image